# Notebook 08 -- Labor-Efficiency Analysis

**Green Slotting: Phase 4 -- Impact Quantification**

This notebook converts Notebook 07's fair (equalized-budget) travel-distance results
into labor-time savings -- the headline operational-impact metric for this project.

**Grounded in the dataset's own published data-descriptor paper** (de Assis, de Paula
Ferreira & Ouhimmou, 2025, *Data in Brief* 61, 111837), which confirms:

1. **Warehouse location: Sherbrooke, Quebec, Canada** (Table, "Data source location").
2. **Picking is done with manual trolleys, not powered vehicles.** The paper states:
   *"levels 1 and 2 for manual picking and levels 3 and 4 as replenishment zones
   accessed by forklifts... operators use trolleys to navigate demand-driven picking
   waves."* Every picking wave in `Picking_Wave.csv` -- the travel distance this
   whole pipeline optimizes -- happens on levels 1-2, by an operator walking and
   pushing a manual trolley.

Because picking is manual, **labor time is the direct, equipment-independent
consequence of travel distance** -- less distance walked while pushing a trolley is
less operator time per wave, full stop, with no equipment assumptions needed. This is
the metric this notebook reports.

**Inputs** (same folder as this notebook):
- `wave_evaluation_results.csv` -- from Notebook 07

**Outputs:**
- `labor_efficiency_results.csv` -- labor-time figures per layout
- `labor_efficiency_comparison.png` -- comparison chart

Runs in a few seconds.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries loaded.")

In [ ]:
import os as _os
# ── Image output folder ─────────────────────────────────────────────────────
# Figures are saved to a "figures/" subfolder in the directory where you run
# this notebook. Change FIGURES_DIR below if you prefer a different path.
FIGURES_DIR = _os.path.join(_os.getcwd(), "figures")
_os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"[INFO] Figures will be saved to: {FIGURES_DIR}")

## 2. Assumptions

**Confirmed facts** (from the dataset's own paper, not assumptions):
- Warehouse location: Sherbrooke, Quebec, Canada
- Picking equipment: manual trolley (operator-walked), levels 1-2 only
- Storage capacity: 18 products/location (already used in Notebook 06 -- directly
  confirmed by the source paper's Table 1)

**Remaining modeling assumption** (clearly labeled, with citation):
- Walking/trolley-pushing speed: 1 m/s -- the source paper doesn't state an exact
  picker speed, so we use the same sourced figure as earlier notebooks (an
  operations-research paper's picker-speed assumption for warehouse simulations;
  full citation in Section 6), reasonable for an operator walking while pushing a
  loaded trolley.

In [ ]:
# --- confirmed facts ---
WAREHOUSE_LOCATION = "Sherbrooke, Quebec, Canada"
PICKING_EQUIPMENT = "manual trolley (operator-walked, levels 1-2)"

# --- labor-time model ---
TROLLEY_SPEED_M_PER_S = 1.0          # walking speed while pushing a loaded trolley

# --- time-period scaling (for the annual extrapolation) ---
WAVES_IN_FULL_DATASET = 9784
WAVES_IN_FAIR_EVALUATION = 1187
DAYS_SPANNED_BY_DATASET = 286

print(f"Warehouse: {WAREHOUSE_LOCATION}")
print(f"Picking equipment: {PICKING_EQUIPMENT}")
print("Assumptions set. See Section 6 for full citations.")

## 3. Load Notebook 07's results

Applied to the **fair (equalized-budget)** distances -- the placement-quality-isolated
comparison from Notebook 07.

In [ ]:
results = pd.read_csv("wave_evaluation_results.csv")
results["fair_km"] = results["fair_distance"] / 1000
results = results.sort_values("fair_km")
print(results[["layout", "fair_distance", "fair_km"]].to_string(index=False))

## 4. Labor-time savings

The headline result for this warehouse: distance walked while pushing a trolley,
converted directly to operator time.

In [ ]:
results["labor_hours"] = (results["fair_distance"] / TROLLEY_SPEED_M_PER_S) / 3600

random_hours = results.loc[results["layout"] == "Random", "labor_hours"].values[0]
results["hours_saved_vs_random"] = random_hours - results["labor_hours"]
results["pct_saved_vs_random"] = 100 * results["hours_saved_vs_random"] / random_hours

scale_to_full_waves = WAVES_IN_FULL_DATASET / WAVES_IN_FAIR_EVALUATION
scale_to_year = 365 / DAYS_SPANNED_BY_DATASET
annual_scale = scale_to_full_waves * scale_to_year
results["est_annual_hours_saved_vs_random"] = results["hours_saved_vs_random"] * annual_scale

print(f"Extrapolation factor (eval sample -> full year): {annual_scale:.2f}x\n")
print(results[["layout", "labor_hours", "hours_saved_vs_random",
                "pct_saved_vs_random", "est_annual_hours_saved_vs_random"]]
      .sort_values("labor_hours").to_string(index=False))

## 5. Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_map = {"GA-ML": "#1d9e75", "GA-Ridge": "#0f6e56", "Random": "#888780",
              "Dedicated": "#d85a30", "Class-Based": "#378add", "Hybrid": "#ba7517"}

ordered = results.sort_values("labor_hours")
axes[0].barh(ordered["layout"], ordered["labor_hours"],
             color=[colors_map[n] for n in ordered["layout"]])
axes[0].set_title(f"Labor-hours for evaluation sample\n({WAVES_IN_FAIR_EVALUATION} waves, fair comparison)")
axes[0].set_xlabel("labor-hours (trolley-walking time)")
axes[0].invert_yaxis()
axes[0].grid(alpha=0.3, axis="x")

ordered2 = results.sort_values("est_annual_hours_saved_vs_random", ascending=False)
ordered2 = ordered2[ordered2["layout"] != "Random"]
axes[1].barh(ordered2["layout"], ordered2["est_annual_hours_saved_vs_random"],
             color=[colors_map[n] for n in ordered2["layout"]])
axes[1].set_title("Estimated annual labor-hours saved vs. Random\n(extrapolated, illustrative)")
axes[1].set_xlabel("labor-hours saved / year")
axes[1].invert_yaxis()
axes[1].grid(alpha=0.3, axis="x")
axes[1].axvline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig(_os.path.join(FIGURES_DIR, "08_labor_efficiency_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
results.to_csv("labor_efficiency_results.csv", index=False)
print("Saved labor_efficiency_results.csv")

ga_ml_hrs = results.loc[results["layout"]=="GA-ML", "hours_saved_vs_random"].values[0]
ga_ml_pct = results.loc[results["layout"]=="GA-ML", "pct_saved_vs_random"].values[0]
ga_ml_annual = results.loc[results["layout"]=="GA-ML", "est_annual_hours_saved_vs_random"].values[0]
ga_ridge_hrs = results.loc[results["layout"]=="GA-Ridge", "hours_saved_vs_random"].values[0]
ga_ridge_pct = results.loc[results["layout"]=="GA-Ridge", "pct_saved_vs_random"].values[0]
ga_ridge_annual = results.loc[results["layout"]=="GA-Ridge", "est_annual_hours_saved_vs_random"].values[0]

print(f"\nHeadline numbers for the paper:")
print(f"  GA-ML    saves {ga_ml_hrs:.1f} labor-hours ({ga_ml_pct:.1f}%) vs Random on the evaluation")
print(f"           sample -- an estimated {ga_ml_annual:,.0f} labor-hours/year if extrapolated")
print(f"           (~{ga_ml_annual/1880:.2f} FTE-years at ~1,880 working hours/year/FTE).")
print(f"  GA-Ridge saves {ga_ridge_hrs:.1f} labor-hours ({ga_ridge_pct:.1f}%) vs Random on the evaluation")
print(f"           sample -- an estimated {ga_ridge_annual:,.0f} labor-hours/year if extrapolated")
print(f"           (~{ga_ridge_annual/1880:.2f} FTE-years at ~1,880 working hours/year/FTE).")

## Done -- Phase 4 complete

**The full chain for this real warehouse:** better demand forecast (LightGBM,
Notebook 01-04) -> better GA-driven slot assignment (Notebook 06) -> genuinely less
picker travel, confirmed on real picking waves with replica-count bias removed
(Notebook 07) -> **measurable labor-time savings** (this notebook) -- the metric that
directly applies to this warehouse's real, manual-trolley picking operation.

**For the paper:** GA-ML and GA-Ridge save an estimated **1,481 and 1,886
labor-hours/year** respectively vs. Random (roughly **0.8-1.0 FTE-years/year**),
extrapolated from a fair, replica-count-controlled sample of real picking waves.
State the extrapolation explicitly as an estimate (Section 4 caveats how it's derived
from a partial wave sample), not a measured full-year figure.

---

## References

1. **Warehouse location (Sherbrooke, Quebec, Canada) and picking equipment (manual
   trolleys, levels 1-2), and storage capacity (18/location):** de Assis, R., de
   Paula Ferreira, W., & Ouhimmou, M. (2025). "Order picking dataset from a
   warehouse of a footwear manufacturing company." *Data in Brief*, 61, 111837.
   https://doi.org/10.1016/j.dib.2025.111837 -- the dataset's own official
   data-descriptor paper (open access, CC BY 4.0).

2. **Warehouse coordinate units (meters):** this project's own `README.txt` data
   dictionary, `Storage_Location.csv` field descriptions (consistent with the above
   source paper's CAD-derived Cartesian mapping).

3. **Trolley/walking travel speed (1 m/s), modeling assumption:** used as the
   picker travel-speed assumption in an operations-research simulation of dynamic
   warehouse order picking. "Deep Reinforcement Learning for Dynamic Order Picking
   in Warehouse Operations," arXiv:2408.01656. https://arxiv.org/pdf/2408.01656
   (The source dataset paper does not itself state an exact picker speed.)

In [ ]:
# ── Extra graphic: FTE-year impact + % savings waterfall ──────────────────
import matplotlib.pyplot as plt
import numpy as np
import os as _os

colors_map = {"GA-ML": "#1d9e75", "GA-Ridge": "#0f6e56", "Random": "#888780",
              "Dedicated": "#d85a30", "Class-Based": "#378add", "Hybrid": "#ba7517"}

non_random = results[results["layout"] != "Random"].copy().sort_values(
    "est_annual_hours_saved_vs_random", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: estimated annual FTE-years saved
FTE_HOURS = 1880
non_random["fte_years"] = non_random["est_annual_hours_saved_vs_random"] / FTE_HOURS
bar_colors = [colors_map.get(n, "#555555") for n in non_random["layout"]]
bars = axes[0].bar(non_random["layout"], non_random["fte_years"],
                   color=bar_colors, edgecolor="white", linewidth=0.5)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Estimated Annual Labor Saving\nvs Random (FTE-years/year)", fontweight="bold")
axes[0].set_ylabel("FTE-years saved / year  (@ 1,880 hrs/FTE)")
axes[0].set_xlabel("")
for bar, val in zip(bars, non_random["fte_years"]):
    ypos = bar.get_height() + 0.01 if val >= 0 else bar.get_height() - 0.04
    axes[0].text(bar.get_x() + bar.get_width()/2, ypos, f"{val:+.2f}",
                 ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[0].grid(alpha=0.3, axis="y"); axes[0].tick_params(axis="x", rotation=10)

# Right: % saving vs Random in the evaluation sample
pct_data = results.sort_values("pct_saved_vs_random", ascending=False)
pct_data = pct_data[pct_data["layout"] != "Random"]
bar_colors2 = [colors_map.get(n, "#555555") for n in pct_data["layout"]]
bars2 = axes[1].bar(pct_data["layout"], pct_data["pct_saved_vs_random"],
                    color=bar_colors2, edgecolor="white")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Labor-Hour Saving vs Random\n(evaluation sample, %)", fontweight="bold")
axes[1].set_ylabel("% labor-hours saved vs Random")
for bar, val in zip(bars2, pct_data["pct_saved_vs_random"]):
    ypos = val + 0.3 if val >= 0 else val - 0.8
    axes[1].text(bar.get_x() + bar.get_width()/2, ypos, f"{val:+.1f}%",
                 ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[1].grid(alpha=0.3, axis="y"); axes[1].tick_params(axis="x", rotation=10)

plt.tight_layout()
plt.savefig(_os.path.join(FIGURES_DIR, "08_fte_and_pct_savings.png"), dpi=150, bbox_inches="tight")
plt.show()